In [1]:
from pipeline_wlog import Pipeline
from utils.logging_setup import setup_latency_logger
from utils.audio_streaming import stream_audio
from utils.audio_preprocessing import preprocess_audio
from threading import Thread
import sounddevice as sd
import time
import os

In [2]:
def run_pipeline(wav_path, input_language="en", output_language="da", min_chunk_size=1000):
    audio = preprocess_audio(wav_path)

    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    # build a filename to distinguish different chunk sizes
    file_name = f"{base_name}_chunk{min_chunk_size}"

    pipeline = Pipeline(input_language=input_language, output_language=output_language, min_chunk_size=min_chunk_size, file_name=file_name, device="mps")
    pipeline.start()

    def print_outputs(queue):
        while True:
            result = queue.get()
            if result is None:
                break
            transcript, translated, audio = result
            sd.play(audio, 16000)
            sd.wait()

    printer_thread = Thread(target=print_outputs, args=(pipeline.output_queue,))
    printer_thread.start()

    for i, chunk in enumerate(stream_audio(audio, frame_ms=200)):
        start_time = time.perf_counter()

        # Log the time when the audio chunk is sent to the pipeline (each sample is enumerated and time is logged) (!OBS: this is samples, not chunks)
        # But the time is only logged for the last sample of each chunk, so it will not log all samples.
        pipeline.audio_queue.put((i, start_time, chunk))

        
    pipeline.stop()
    printer_thread.join()

In [4]:
dk_data = ["data/danish/dk_speaker_1.wav", 
           "data/danish/dk_speaker_2.wav", 
           "data/danish/dk_speaker_3.wav", 
           "data/danish/dk_speaker_4.wav", 
           "data/danish/dk_speaker_5.wav",
           "data/danish/dk_speaker_6.wav",
           "data/danish/dk_speaker_7.wav",
           "data/danish/dk_speaker_8.wav",
           "data/danish/dk_speaker_9.wav",
           "data/danish/dk_speaker_10.wav"]

en_data = ["data/english/speaker_1_final.wav",
           "data/english/speaker_2_final.wav",
           "data/english/speaker_3_final.wav",
           "data/english/speaker_4_final.wav",
           "data/english/speaker_5_final.wav",
           "data/english/speaker_6_final.wav",
           "data/english/speaker_7_final.wav",
           "data/english/speaker_8_final.wav",
           "data/english/speaker_9_final.wav",
           "data/english/speaker_10_final.wav"]

chunk_sizes_testing = [3000, 3500, 4000, 4500, 5000, 5500, 6000]

--- 

## Danish to English

In [ ]:
input_language = "da"
output_language = "en"

for chunk_size in chunk_sizes_testing:
    for input_file in dk_data:
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        # build a filename to distinguish different chunk sizes
        file_name = f"{base_name}_chunk{chunk_size}"

        setup_latency_logger(file_name=file_name)
        results = run_pipeline(input_file, input_language=input_language, output_language=output_language, min_chunk_size=chunk_size)
        print("logging complete")

---

## English to Danish

In [ ]:
input_language = "en"
output_language = "da"

for chunk_size in chunk_sizes_testing:
    for input_file in en_data:
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        # build a filename to distinguish different chunk sizes
        file_name = f"{base_name}_chunk{chunk_size}"

        setup_latency_logger(file_name=file_name)
        results = run_pipeline(input_file, input_language=input_language, output_language=output_language, min_chunk_size=chunk_size)
        print("logging complete")